# Synthetic Experiments with SSSD

This notebook demonstrates how to run **SSSD (Similar Scores, Similar Data)** on synthetic anomaly detection datasets.

A larger SSSD value indicates a stronger violation of score-data consistency and therefore a higher probability of unreliable detection.


## Step 1: Environment setup

The synthetic datasets should be placed under the `dataset/artificial` directory.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score
from pyod.utils.utility import precision_n_scores, standardizer

from pyod.models.iforest import IForest
from pyod.models.lof import LOF
from pyod.models.ocsvm import OCSVM
from pyod.models.knn import KNN
from pyod.models.hbos import HBOS
from pyod.models.abod import ABOD
from pyod.models.loda import LODA
from pyod.models.pca import PCA

# Make sure SSSD.py in the project root can be imported.
# This works whether the notebook is launched from the project root
# or from the notebooks/ directory.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
sys.path.insert(0, str(PROJECT_ROOT))

from SSSD import SSSD

# print(f"Project root: {PROJECT_ROOT}")

## 2. Configuration


In [2]:
DATA_DIR = PROJECT_ROOT / "dataset"/"artificial"

# You can add or remove datasets depending on which files are available locally.
DATASETS = [
    "DataSet1(2D)",
    "DataSet2(2D)",
    "DataSet3(2D)",
    "DataSet4(2D)",
    "DataSet5(2D)",
]

DETECTORS = [
    ("LODA",LODA()),
    ("ABOD",ABOD()),
    ("iForest",IForest()),
    ('kNN',KNN()),
    ('LOF',LOF()),
    ("HBOS",HBOS()),
    ("OCSVM",OCSVM()),
    ("PCA",PCA())

]

# print(f"Data directory: {DATA_DIR}")

## Step 3: Helper functions

The following helper functions keep the main experiment loop clean.

- `load_synthetic_dat`: loads a synthetic 2D dataset.
- `normalize_scores`: maps anomaly scores to `[0, 1]`.
- `evaluate_detector`: trains one detector and computes both external metrics and SSSD.


In [3]:
def load_synthetic_dat(file_path):
    """Load a synthetic 2D dataset"""
    datafile = file_path
    gtfile = str(file_path).replace('(2D)', 'GT')
    X = np.loadtxt(datafile,delimiter=' ')
    y = np.loadtxt(gtfile,delimiter=' ')

    # Standardization is commonly used for distance-based anomaly detectors.
    X = standardizer(X)
    return X, y
    
def normalize_scores(scores):
    """Normalize anomaly scores to [0, 1]"""
    scores = np.asarray(scores, dtype=float)
    score_min = np.min(scores)
    score_max = np.max(scores)
    denom = score_max - score_min
    if denom == 0:
        return np.zeros_like(scores, dtype=float)
    return (scores - score_min) / denom


def evaluate_detector(name, detector, X, y, sssd_k=5, sssd_m=5):
    """Train one anomaly detector and compute external metrics plus SSSD."""
    detector.fit(X)

    # PyOD detectors return larger decision_function values for more anomalous samples.
    scores = detector.decision_function(X)
    scores = normalize_scores(scores)

    # External metrics require labels and are used here only for validation/analysis.
    roc_auc = roc_auc_score(y, scores)
    avg_precision = average_precision_score(y, scores)
    precision_at_k = precision_n_scores(y, scores)

    # SSSD is an internal evaluation score computed from X and anomaly scores only.
    sssd = SSSD(X, scores, k=sssd_k, m=sssd_m)
    _, final_sssd_sum, _ = sssd.get_final_SSSD()
    _, score_sssd_sum, _ = sssd.get_Score_SSSD()
    _, distance_sssd_sum, _ = sssd.get_Distance_SSSD()

    return {
        "Detector": name,
        "ROC-AUC": roc_auc,
        "AP": avg_precision,
        "Precision@k": precision_at_k,
        "Final SSSD": final_sssd_sum,
        "Score-SSSD": score_sssd_sum,
        "Distance-SSSD": distance_sssd_sum,
    }

## Step 4: Run synthetic experiments

For each synthetic dataset, we train the selected detectors and compute both external metrics and SSSD.

Note that ROC-AUC, AP, and Precision@k are computed only for analysis because they require ground-truth labels. In contrast, SSSD itself does not use labels.


In [4]:
all_results = []

for dataset_name in DATASETS:
    file_path = DATA_DIR / dataset_name
    X, y = load_synthetic_dat(file_path)
    print(f"Dataset: {dataset_name}")
    print(f"Shape: {X.shape}, anomaly ratio: {np.mean(y):.4f}")

    for detector_name, detector in DETECTORS:
        try:
            result = evaluate_detector(detector_name, detector, X, y)
            result["Dataset"] = dataset_name.replace(".mat", "")
            all_results.append(result)
            print(
                f"  {detector_name:<8} "
                f"ROC-AUC={result['ROC-AUC']:.4f}, "
                f"AP={result['AP']:.4f}, "
                f"Final SSSD={result['Final SSSD']:.4f}"
            )
        except Exception as exc:
            print(f"  {detector_name:<8} failed: {exc}")

results_df = pd.DataFrame(all_results)
results_df

Dataset: DataSet1(2D)
Shape: (1000, 2), anomaly ratio: 0.1000
  LODA     ROC-AUC=0.9997, AP=0.9962, Final SSSD=5.7424
  ABOD     ROC-AUC=0.9855, AP=0.8851, Final SSSD=3.8599
  iForest  ROC-AUC=0.9982, AP=0.9874, Final SSSD=4.9583
  kNN      ROC-AUC=0.9986, AP=0.9906, Final SSSD=3.6795
  LOF      ROC-AUC=0.5881, AP=0.2242, Final SSSD=14.8563
  HBOS     ROC-AUC=0.9951, AP=0.9600, Final SSSD=8.4795
  OCSVM    ROC-AUC=0.9994, AP=0.9959, Final SSSD=3.5184
  PCA      ROC-AUC=0.9901, AP=0.9682, Final SSSD=5.3597
Dataset: DataSet2(2D)
Shape: (1000, 2), anomaly ratio: 0.1000
  LODA     ROC-AUC=0.7137, AP=0.3875, Final SSSD=8.0844
  ABOD     ROC-AUC=0.8909, AP=0.5444, Final SSSD=4.8720
  iForest  ROC-AUC=0.9260, AP=0.6731, Final SSSD=4.8360
  kNN      ROC-AUC=0.9134, AP=0.6286, Final SSSD=3.7368
  LOF      ROC-AUC=0.8368, AP=0.5108, Final SSSD=6.3867
  HBOS     ROC-AUC=0.8442, AP=0.4742, Final SSSD=7.5868
  OCSVM    ROC-AUC=0.8988, AP=0.6604, Final SSSD=5.5775
  PCA      ROC-AUC=0.7068, AP=0.464

,Detector,ROC-AUC,AP,Precision@k,Final SSSD,Score-SSSD,Distance-SSSD,Dataset
0,LODA,0.999661,0.996153,0.979798,5.742404,3.033866,8.450943,DataSet1(2D)
1,ABOD,0.985533,0.885106,0.850000,3.859862,2.231036,5.488689,DataSet1(2D)
2,iForest,0.998200,0.987406,0.950000,4.958293,3.060497,6.856090,DataSet1(2D)
3,kNN,0.998556,0.990572,0.960000,3.679534,4.555626,2.803441,DataSet1(2D)
4,LOF,0.588078,0.224241,0.200000,14.856267,6.396461,23.316073,DataSet1(2D)
5,HBOS,0.995083,0.960026,0.868687,8.479495,8.298680,8.660311,DataSet1(2D)
6,OCSVM,0.999411,0.995917,0.970000,3.518416,1.492393,5.544440,DataSet1(2D)
7,PCA,0.990067,0.968243,0.910000,5.359746,1.292565,9.426927,DataSet1(2D)
8,LODA,0.713678,0.387490,0.575000,8.084431,7.590484,8.578378,DataSet2(2D)
9,ABOD,0.890944,0.544364,0.550000,4.872046,5.039656,4.704437,DataSet2(2D)


## Step 5: Summarize detector-level results

The table below provides a compact summary of datasets, with values sorted in descending order of AP.

In [5]:
if len(results_df) > 0:
    summary_df = (
        results_df
        .groupby("Detector")[["ROC-AUC", "AP", "Precision@k", "Final SSSD", "Score-SSSD", "Distance-SSSD"]]
        .mean()
        .sort_values("AP", ascending=False)
    )
    display(summary_df.round(4))
else:
    print("No results to summarize. Please add datasets under the data/ directory and rerun the notebook.")

,ROC-AUC,AP,Precision@k,Final SSSD,Score-SSSD,Distance-SSSD
Detector,,,,,,
kNN,0.9498,0.8507,0.8160,3.7229,4.8478,2.5979
ABOD,0.9329,0.7744,0.7440,5.8891,6.7972,4.9810
iForest,0.8979,0.7574,0.7040,5.8385,4.0689,7.6080
OCSVM,0.8610,0.7465,0.6940,5.2210,2.6651,7.7768
LOF,0.8438,0.6387,0.5860,7.6415,3.6608,11.6221
HBOS,0.8562,0.6253,0.5949,8.2246,7.2964,9.1529
PCA,0.7786,0.5723,0.5300,7.7670,2.6635,12.8705
LODA,0.7671,0.5547,0.5704,8.4158,7.1858,9.6459
